# 01 · Data preparation (CIC-IDS-2017, 5 classes)

Loads the raw CIC-IDS-2017 CSV files, keeps the **51 features** used by the ASTRA API, maps the raw
labels to **5 classes** (benign, bruteforce, ddos, dos, portscan), cleans the data, makes a stratified
80/20 split and fits a `RobustScaler` on the training split only.

**Input:** the original `MachineLearningCVE` folder (8 CSV files).
Set it with the environment variable `ASTRA_DATA_DIR` or edit `DATA_DIR` below.
**Output:** `astra_run/splits.joblib`, `astra_run/artifacts/scaler.pkl`, `label_encoder.pkl`, `data_meta.json`.

Run the notebooks in order: 01 → 02 → 03 → 04.

In [ ]:
import os, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

WORK_DIR = Path(os.environ.get("ASTRA_WORK_DIR", "astra_run"))   # shared by all four notebooks
ART_DIR = WORK_DIR / "artifacts"
SEED = 42
CLASSES = ["benign", "bruteforce", "ddos", "dos", "portscan"]   # alphabetical = LabelEncoder order
ART_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path(os.environ.get("ASTRA_DATA_DIR", r"C:\path\to\MachineLearningCVE"))
SMOKE = os.environ.get("ASTRA_SMOKE", "0") == "1"      # True = 5% sample, for a quick dry run only
TEST_SIZE = 0.2
MAX_TRAIN_PER_CLASS = 0         # 0 = use all training rows; e.g. 200_000 if RAM is limited (test set is never capped)
DROP_DUPLICATES = True          # drop exact duplicate rows before splitting (prevents train/test leakage)
print("DATA_DIR:", DATA_DIR, "| SMOKE:", SMOKE)

In [ ]:
EXPECTED_COLS = [
    "Flow_Duration", "Total_Fwd_Packets", "Total_Length_of_Fwd_Packets",
    "Fwd_Packet_Length_Max", "Fwd_Packet_Length_Min", "Fwd_Packet_Length_Mean",
    "Bwd_Packet_Length_Max", "Bwd_Packet_Length_Min", "Flow_Bytes/s", "Flow_Packets/s",
    "Flow_IAT_Mean", "Flow_IAT_Std", "Flow_IAT_Max", "Flow_IAT_Min", "Fwd_IAT_Mean",
    "Fwd_IAT_Std", "Fwd_IAT_Min", "Bwd_IAT_Total", "Bwd_IAT_Mean", "Bwd_IAT_Std",
    "Bwd_IAT_Max", "Bwd_IAT_Min", "Fwd_PSH_Flags", "Fwd_Header_Length",
    "Bwd_Header_Length", "Bwd_Packets/s", "Min_Packet_Length", "Max_Packet_Length",
    "Packet_Length_Mean", "Packet_Length_Variance", "FIN_Flag_Count", "RST_Flag_Count",
    "PSH_Flag_Count", "ACK_Flag_Count", "URG_Flag_Count", "Down/Up_Ratio",
    "Fwd_Avg_Bytes/Bulk", "Fwd_Avg_Packets/Bulk", "Fwd_Avg_Bulk_Rate",
    "Bwd_Avg_Bytes/Bulk", "Bwd_Avg_Packets/Bulk", "Bwd_Avg_Bulk_Rate",
    "Init_Win_bytes_forward", "Init_Win_bytes_backward", "act_data_pkt_fwd",
    "min_seg_size_forward", "Active_Mean", "Active_Std", "Active_Max",
    "Active_Min", "Idle_Std",
]
assert len(EXPECTED_COLS) == 51

LABEL_MAP = {                       # raw label (lower-case) -> ASTRA class; anything else is dropped
    "benign": "benign",
    "dos hulk": "dos", "dos goldeneye": "dos", "dos slowloris": "dos", "dos slowhttptest": "dos",
    "ddos": "ddos",
    "ftp-patator": "bruteforce", "ssh-patator": "bruteforce",
    "portscan": "portscan",
}
norm = lambda c: str(c).strip().replace(" ", "_")
KEEP = set(EXPECTED_COLS) | {"Label"}

## 1. Load the CSV files

In [ ]:
files = sorted(Path(DATA_DIR).glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No .csv files in {DATA_DIR}. Set ASTRA_DATA_DIR to the MachineLearningCVE folder.")
frames = []
for f in files:
    # latin-1: one CIC-IDS-2017 file has a non-UTF-8 dash in the web-attack label names
    df = pd.read_csv(f, usecols=lambda c: norm(c) in KEEP, encoding="latin-1", low_memory=False)
    df.columns = [norm(c) for c in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]
    missing = [c for c in EXPECTED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{f.name}: {len(missing)} of 51 features missing, e.g. {missing[:5]}")
    frames.append(df)
    print(f"{f.name:60s} {len(df):>9,} rows")
raw = pd.concat(frames, ignore_index=True)
rows_read = len(raw)
print("total rows read:", f"{rows_read:,}")

## 2. Map labels and clean

In [ ]:
raw_label = raw.pop("Label").astype(str).str.strip().str.lower()
print("Raw label distribution:")
display(raw_label.value_counts().to_frame("rows"))

mapped = raw_label.map(LABEL_MAP)
dropped_labels = raw_label[mapped.isna()].value_counts()
print("Out-of-scope labels dropped:", int(dropped_labels.sum()))

keep = mapped.notna()
X = raw.loc[keep, EXPECTED_COLS].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
y = mapped[keep]
ok = X.notna().all(axis=1)
n_nan = int((~ok).sum())
X, y = X[ok], y[ok]

n_dup = 0
if DROP_DUPLICATES:
    dup = pd.concat([X, y.rename("y")], axis=1).duplicated(keep="first")
    n_dup = int(dup.sum())
    X, y = X[~dup], y[~dup]
X, y = X.reset_index(drop=True), y.reset_index(drop=True)
del raw, frames

if SMOKE:
    from sklearn.model_selection import train_test_split
    X, _, y, _ = train_test_split(X, y, train_size=0.05, stratify=y, random_state=SEED)
    X, y = X.reset_index(drop=True), y.reset_index(drop=True)

class_counts = y.value_counts().reindex(CLASSES).fillna(0).astype(int)
print(f"NaN/Inf rows dropped: {n_nan:,} | duplicate rows dropped: {n_dup:,} | final rows: {len(X):,}")
display(class_counts.to_frame("rows"))
assert class_counts.min() >= 10, "A class has fewer than 10 rows - check DATA_DIR"

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
class_counts.plot.bar(ax=ax, log=True, color="#2a6fbb")
ax.set_ylabel("rows (log scale)"); ax.set_title("Class distribution after cleaning")
for i, v in enumerate(class_counts.values): ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()

## 3. Stratified split and scaling (scaler fitted on the training split only)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler

le = LabelEncoder().fit(CLASSES)
assert list(le.classes_) == CLASSES
y_enc = le.transform(y)

X_tr, X_te, y_tr, y_te = train_test_split(X, y_enc, test_size=TEST_SIZE, stratify=y_enc, random_state=SEED)

if MAX_TRAIN_PER_CLASS > 0:
    rng = np.random.RandomState(SEED)
    idx = np.concatenate([rng.choice(np.where(y_tr == k)[0], min(MAX_TRAIN_PER_CLASS, int((y_tr == k).sum())), replace=False)
                          for k in np.unique(y_tr)])
    X_tr, y_tr = X_tr.iloc[idx], y_tr[idx]

scaler = RobustScaler()
X_tr_s = scaler.fit_transform(X_tr)           # fitted on a DataFrame -> stores feature_names_in_
X_te_s = scaler.transform(X_te)
print(f"train {X_tr_s.shape} | test {X_te_s.shape}")
print("train class counts:", dict(zip(CLASSES, np.bincount(y_tr, minlength=5))))
print("test  class counts:", dict(zip(CLASSES, np.bincount(y_te, minlength=5))))

## 4. Save

In [ ]:
joblib.dump({"X_train_raw": X_tr.values, "X_test_raw": X_te.values,
             "X_train": X_tr_s, "X_test": X_te_s, "y_train": y_tr, "y_test": y_te},
            WORK_DIR / "splits.joblib")
joblib.dump(scaler, ART_DIR / "scaler.pkl")
joblib.dump(le, ART_DIR / "label_encoder.pkl")

meta = {"dataset": "CIC-IDS-2017", "files": [f.name for f in files], "features": EXPECTED_COLS, "classes": CLASSES,
        "rows_read": int(rows_read), "dropped_out_of_scope_labels": {k: int(v) for k, v in dropped_labels.items()},
        "rows_dropped_nan_inf": n_nan, "rows_dropped_duplicates": n_dup,
        "class_counts_after_cleaning": {k: int(v) for k, v in class_counts.items()},
        "train_rows": int(len(X_tr_s)), "test_rows": int(len(X_te_s)),
        "split": {"test_size": TEST_SIZE, "stratified": True, "seed": SEED,
                  "max_train_per_class": MAX_TRAIN_PER_CLASS, "smoke": SMOKE},
        "scaler": "RobustScaler (fit on train only)", "resampling": "none"}
(WORK_DIR / "data_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("saved to", WORK_DIR.resolve())
if SMOKE: print("SMOKE run - do not use these results in the paper.")